In [110]:
# Basic imports
import os
import uuid
from tqdm import tqdm

# Third party core
import boto3
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from torchvision import transforms

# Sagemaker imports
import sagemaker
from sagemaker.deserializers import JSONDeserializer
from sagemaker.model_monitor import (
    CronExpressionGenerator,
    DataCaptureConfig,
    DatasetFormat,
    EndpointInput,
    ModelQualityMonitor,
)
from sagemaker.predictor import Predictor
from sagemaker.pytorch import PyTorchModel
from sagemaker.serializers import NumpySerializer, IdentitySerializer

In [111]:
%store -r

In [112]:
%store

Stored variables and their in-db values:
bucket                                     -> 'sagemaker-us-east-1-298748835671'
database_name                              -> 'cat_landmarking'
ingest_create_athena_db_passed             -> True
ingestion_completed                        -> True
landmarks_table                            -> 'cat_annotations'
manifest_table                             -> 'image_manifest'
project_prefix                             -> 'cat-landmarks-project'
s3_athena_results_dir                      -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_processed_cats_prefix                   -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_processed_combined_prefix               -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_raw_cats_prefix                         -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_raw_noncats_prefix                      -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_staging_dir                

In [113]:
bucket = bucket
project_prefix = project_prefix
database_name = database_name
manifest_table = manifest_table
s3_staging_dir = s3_staging_dir

s3 = boto3.client("s3")
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session._region_name
boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(service_name="sagemaker", 
                                       region_name=region)

# Monitor image
monitor_image_uri = sagemaker.image_uris.retrieve(framework="model-monitor", region=region)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [114]:
# Endpoint
endpoint_id = uuid.uuid4()
kp_endpoint_name = f"keypoint-regression-{endpoint_id}"
kp_model_url = "s3://sagemaker-us-east-1-298748835671/cat-landmarks-project/models/model_artifacts/pipelines-z5wekhj55a97-TrainKeypointModel-urHLZKUIGv/output/model.tar.gz"
kp_pipeline_path = "KeypointPipeline/z5wekhj55a97/KeypointPreprocessingEval/output/test"
annotations_key = f"{kp_pipeline_path}/annotations.parquet"

cls_endpoint_name = f"cls-{endpoint_id}"
cls_model_url = "s3://sagemaker-us-east-1-298748835671/cat-landmarks-project/models/artifacts/model_cls_1ed1fb2a-aad1-4019-a688-ecfa2c11da29.tar.gz"

# Creating Keypoint model reference
kp_model = PyTorchModel(
    model_data=kp_model_url,
    role=role,
    framework_version="2.0",
    py_version="py310",
    source_dir="src/keypoint_regression",
    entry_point="inference.py",
    sagemaker_session=sagemaker_session,
)

# Creating Classification model reference
cls_model = PyTorchModel(
    model_data=cls_model_url,
    role=role,
    framework_version="2.0",
    py_version="py310",
    source_dir="src/classification",
    entry_point="inference.py",
    sagemaker_session=sagemaker_session,
)

## Model Monitor set up

In [115]:
data_capture_prefix = f"{project_prefix}/datacapture/{endpoint_id}"
s3_capture_upload_path = f"s3://{bucket}/{data_capture_prefix}"

# Keypoint model configs
kp_data_capture_config = DataCaptureConfig(
    enable_capture=True, 
    sampling_percentage=100, 
    destination_s3_uri=f"{s3_capture_upload_path}/keypoint_regressor"
)

# Deploying Keypoint model
kp_model.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.xlarge",
    endpoint_name=kp_endpoint_name,
    data_capture_config=kp_data_capture_config,
)

INFO:sagemaker:Repacking model artifact (s3://sagemaker-us-east-1-298748835671/cat-landmarks-project/models/model_artifacts/pipelines-z5wekhj55a97-TrainKeypointModel-urHLZKUIGv/output/model.tar.gz), script artifact (src/keypoint_regression), and dependencies ([]) into single tar.gz file located at s3://sagemaker-us-east-1-298748835671/pytorch-inference-2026-02-23-02-16-49-820/model.tar.gz. This may take some time depending on model size...
INFO:sagemaker:Creating model with name: pytorch-inference-2026-02-23-02-17-02-660
INFO:sagemaker:Creating endpoint-config with name keypoint-regression-e41ed1f6-f8f9-4adc-9afd-335ab04e08bd
INFO:sagemaker:Creating endpoint with name keypoint-regression-e41ed1f6-f8f9-4adc-9afd-335ab04e08bd


--------!

In [116]:
# Classifier configs
cls_data_capture_config = DataCaptureConfig(
    enable_capture=True, 
    sampling_percentage=100, 
    destination_s3_uri=f"{s3_capture_upload_path}/classifier"
)

# Deploying Classification model
cls_model.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.xlarge",
    endpoint_name=cls_endpoint_name,
    data_capture_config=cls_data_capture_config,
)

INFO:sagemaker:Repacking model artifact (s3://sagemaker-us-east-1-298748835671/cat-landmarks-project/models/artifacts/model_cls_1ed1fb2a-aad1-4019-a688-ecfa2c11da29.tar.gz), script artifact (src/classification), and dependencies ([]) into single tar.gz file located at s3://sagemaker-us-east-1-298748835671/pytorch-inference-2026-02-23-02-21-35-226/model.tar.gz. This may take some time depending on model size...


INFO:sagemaker:Creating model with name: pytorch-inference-2026-02-23-02-21-38-728
INFO:sagemaker:Creating endpoint-config with name cls-e41ed1f6-f8f9-4adc-9afd-335ab04e08bd
INFO:sagemaker:Creating endpoint with name cls-e41ed1f6-f8f9-4adc-9afd-335ab04e08bd


--------!

In [117]:
# Keypoint predictor
kp_predictor = Predictor(
    endpoint_name=kp_endpoint_name, 
    sagemaker_session=sagemaker_session,
    serializer=NumpySerializer(),
    deserializer=JSONDeserializer(),
)

# Classification predictor
cls_predictor = Predictor(
    endpoint_name=cls_endpoint_name, 
    sagemaker_session=sagemaker_session,
    serializer=IdentitySerializer("image/jpeg"),
    deserializer=JSONDeserializer(),
)

# Creating Baseline Job
## Keypoint model baselining

In [118]:
baselining_path = Path("./data/baselining")
if not baselining_path.exists():
    os.makedirs(baselining_path)

ANNOTATIONS  = "./data/baselining/annotations.pqt"
NUM_IMAGES   = 100 

KP_NAMES = [
    "left_eye", "right_eye", "mouth",
    "left_ear_1", "left_ear_2", "left_ear_3",
    "right_ear_1", "right_ear_2", "right_ear_3",
]

KP_COLS_NORM = [f"{kp}_{axis}_norm" for kp in KP_NAMES for axis in ("x", "y")]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

TARGET_W, TARGET_H = 224, 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

s3.download_file(bucket, annotations_key, ANNOTATIONS)
df = pd.read_parquet(ANNOTATIONS)
df.head()
sample = df.sample(n=NUM_IMAGES)

In [119]:
# Generate predictions for baselining job     
raw_results = []
baselining_df = []

for _, row in sample.iterrows():
    # Load and resize image from S3
    key = f"{kp_pipeline_path}/{row["image"]}"
    obj = s3.get_object(Bucket=bucket, Key=key)
    img = Image.open(obj["Body"]).convert("RGB")
    w0, h0 = img.size
    img_resized = img.resize((TARGET_W, TARGET_H), Image.BILINEAR)

    # Run inference
    tensor = transform(img_resized).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = kp_predictor.predict(tensor)
    
    # Compute MSE for this image
    gt = row.drop("image")
    
    pred = [{f"{key}_{k}_norm_pred": pred[key][k] for k in pred[key]} for key in pred.keys()]
    pred = pd.Series({k: v for d in pred for k, v in d.items()})
    
    mse = np.mean((pred.to_numpy() - gt.to_numpy()) ** 2)
    baselining_df.append(mse)
    raw_results.append({**dict(gt), **dict(pred)})
    
raw_results = pd.DataFrame(raw_results)
baselining_df = pd.DataFrame({"prediction": baselining_df})
baselining_df["label"] = baselining_df["prediction"].apply(lambda _: 0)
baselining_df.head()

,prediction,label
0,0.002479,0
1,0.000806,0
2,0.000536,0
3,0.000470,0
4,0.002641,0


In [120]:
local_baseline_path = "./data/baselining/baseline_with_predictions.csv"
baselining_df.to_csv(local_baseline_path)

s3_baselining_upload_path = f"{data_capture_prefix}/baseline/baseline_with_predictions.csv"
s3.upload_file(
    local_baseline_path, 
    bucket,
    s3_baselining_upload_path
    )

## Classification model baselining

In [121]:
# Downloading training records
os.makedirs("data/processed/combined/training_manifests/", exist_ok=True)

local_training_manifest_path = "data/processed/combined/training_manifests/training_manifest_021626.pqt"
training_manifest_path = f"{project_prefix}/{local_training_manifest_path}"

s3.download_file(
	bucket,
 	training_manifest_path,
	local_training_manifest_path
)

training_manifest = pd.read_parquet(local_training_manifest_path)
training_manifest.head()

,remote_path,label,split
0,s3://sagemaker-us-east-1-298748835671/cat-land...,0,train
1,s3://sagemaker-us-east-1-298748835671/cat-land...,0,validation
2,s3://sagemaker-us-east-1-298748835671/cat-land...,0,train
3,s3://sagemaker-us-east-1-298748835671/cat-land...,0,train
4,s3://sagemaker-us-east-1-298748835671/cat-land...,0,train


In [122]:
label_map = dict(zip(training_manifest["remote_path"], training_manifest["label"]))
test_uris = training_manifest.loc[training_manifest["split"] == "test", "remote_path"]

rows = []
for uri in tqdm(test_uris, desc="Performing Batch Inference", unit="image"):
    label = label_map[uri]
    key = uri.replace(f"s3://{bucket}/", "")
    
    try:
        payload = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
        result = cls_predictor.predict(payload)
        pred = result["predicted_class"]
        rows.append({"uri": uri, "label": label, "pred": pred, "error": None})
    except Exception as e:
        rows.append({"uri": uri, "label": label, "pred": None, "error": str(e)})

test_df = pd.DataFrame(rows)
print(f"Completed: {test_df['pred'].notna().sum()}/{len(test_df)} successful")

Performing Batch Inference:   0%|          | 0/2522 [00:00<?, ?image/s]

Performing Batch Inference: 100%|██████████| 2522/2522 [04:03<00:00, 10.37image/s]

Completed: 2522/2522 successful


In [123]:
cls_local_baseline_path = "./data/baselining/cls_baseline_with_predictions.csv"
test_df[["label","pred"]].to_csv(cls_local_baseline_path, index=False)

s3_cls_baselining_upload_path = f"{data_capture_prefix}/baseline/cls_baseline_with_predictions.csv"
s3.upload_file(
    cls_local_baseline_path, 
    bucket,
    s3_cls_baselining_upload_path
    )

# Creating Model Quality Monitors

## Keypoint Model baselining

In [124]:
kp_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session,
)

kp_baseline_job_name = f"keypoint-baseline-job-{endpoint_id}"
job = kp_monitor.suggest_baseline(
    job_name = kp_baseline_job_name,
    baseline_dataset=f"s3://{bucket}/{s3_baselining_upload_path}",
    dataset_format=DatasetFormat.csv(header=True),
    problem_type="Regression",
    inference_attribute="prediction",
    ground_truth_attribute="label",
    output_s3_uri=f"s3://{bucket}/{project_prefix}/{data_capture_prefix}/baseline/results",
)

job.wait(logs=False)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating processing-job with name keypoint-baseline-job-e41ed1f6-f8f9-4adc-9afd-335ab04e08bd


...........................................................!

In [125]:
kp_baseline_job = kp_monitor.latest_baselining_job

In [126]:
pd.DataFrame(kp_baseline_job.baseline_statistics().body_dict["regression_metrics"]).T

,value,standard_deviation
mae,0.003286,NaN
mse,0.000031,NaN
rmse,0.005582,NaN
r2,-Infinity,None


In [127]:
pd.DataFrame(kp_baseline_job.suggested_constraints().body_dict["regression_constraints"]).T

,threshold,comparison_operator
mae,0.003286,GreaterThanThreshold
mse,0.000031,GreaterThanThreshold
rmse,0.005582,GreaterThanThreshold
r2,-Infinity,LessThanThreshold


## Classifier baselining

In [128]:
cls_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session,
)

cls_baseline_job_name = f"cls-baseline-job-{endpoint_id}1"
job = cls_monitor.suggest_baseline(
    job_name = cls_baseline_job_name,
    baseline_dataset=f"s3://{bucket}/{s3_cls_baselining_upload_path}",
    dataset_format=DatasetFormat.csv(header=True),
    problem_type="BinaryClassification",
    inference_attribute="pred",
    ground_truth_attribute="label",
    output_s3_uri=f"s3://{bucket}/{project_prefix}/{data_capture_prefix}/baseline/results",
)

job.wait(logs=False)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating processing-job with name cls-baseline-job-e41ed1f6-f8f9-4adc-9afd-335ab04e08bd1


...........................................................!

In [129]:
cls_baseline_job = cls_monitor.latest_baselining_job

In [130]:
pd.DataFrame(cls_baseline_job.suggested_constraints().body_dict["binary_classification_constraints"]).T

,threshold,comparison_operator
recall,0.964965,LessThanThreshold
precision,0.98167,LessThanThreshold
accuracy,0.978985,LessThanThreshold
true_positive_rate,0.964965,LessThanThreshold
true_negative_rate,0.988181,LessThanThreshold
false_positive_rate,0.011819,GreaterThanThreshold
false_negative_rate,0.035035,GreaterThanThreshold
f0_5,0.978283,LessThanThreshold
f1,0.973246,LessThanThreshold
f2,0.96826,LessThanThreshold


# Setting up Monitoring Schedule

In [131]:
kp_monitor.create_monitoring_schedule(
    monitor_schedule_name="keypoint-model-quality-schedule",
    endpoint_input=EndpointInput(
        endpoint_name=kp_endpoint_name,
        destination="/opt/ml/processing/input/endpoint",
        inference_attribute="mse",
    ),
    ground_truth_input=f"s3://{bucket}/{project_prefix}/ground-truth/",
    problem_type="Regression",
    output_s3_uri=f"s3://{bucket}/{project_prefix}/monitoring/results",
    schedule_cron_expression=CronExpressionGenerator.hourly(),
)

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: keypoint-model-quality-schedule


In [132]:
cls_monitor.create_monitoring_schedule(
    monitor_schedule_name="cls-model-quality-schedule",
    endpoint_input=EndpointInput(
        endpoint_name=cls_endpoint_name,
        destination="/opt/ml/processing/input/endpoint",
        inference_attribute="mse",
    ),
    ground_truth_input=f"s3://{bucket}/{project_prefix}/ground-truth/",
    problem_type="BinaryClassification",
    output_s3_uri=f"s3://{bucket}/{project_prefix}/monitoring/results",
    schedule_cron_expression=CronExpressionGenerator.hourly(),
)

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: cls-model-quality-schedule


# Creating CloudWatch Monitor

In [133]:
cw_client = boto3.client("cloudwatch", region_name=region)

cw_client.put_metric_alarm(
    AlarmName="KeypointModel-MSE-Violation",
    AlarmDescription="Triggers when model MSE violates baseline constraints",
    Namespace="aws/sagemaker/Endpoints/data-metrics",
    MetricName="constraint_violations",
    Dimensions=[
        {"Name": "Endpoint", "Value": kp_endpoint_name},
        {"Name": "MonitoringSchedule", "Value": "keypoint-model-quality-schedule"},
    ],
    Statistic="Sum",
    Period=3600,   
    EvaluationPeriods=1,
    Threshold=1,        
    ComparisonOperator="GreaterThanOrEqualToThreshold",
    TreatMissingData="notBreaching",
)

{'ResponseMetadata': {'RequestId': 'af0247ce-9ea2-45cf-b074-13919a3db857',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'af0247ce-9ea2-45cf-b074-13919a3db857',
   'content-type': 'text/xml',
   'content-length': '214',
   'date': 'Mon, 23 Feb 2026 02:40:35 GMT'},
  'RetryAttempts': 0}}

In [134]:
cw_client = boto3.client("cloudwatch", region_name=region)

cw_client.put_metric_alarm(
    AlarmName="ClsModel-Accuracy-Violation",
    AlarmDescription="Triggers when model Accuracy violates baseline constraints",
    Namespace="aws/sagemaker/Endpoints/data-metrics",
    MetricName="constraint_violations",
    Dimensions=[
        {"Name": "Endpoint", "Value": cls_endpoint_name},
        {"Name": "MonitoringSchedule", "Value": "cls-model-quality-schedule"},
    ],
    Statistic="Sum",
    Period=3600,   
    EvaluationPeriods=1,
    Threshold=1,        
    ComparisonOperator="LessThanThreshold",
    TreatMissingData="notBreaching",
)

{'ResponseMetadata': {'RequestId': 'e6d3858a-d8e6-4691-9847-065283ab5a4b',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'e6d3858a-d8e6-4691-9847-065283ab5a4b',
   'content-type': 'text/xml',
   'content-length': '214',
   'date': 'Mon, 23 Feb 2026 02:40:35 GMT'},
  'RetryAttempts': 0}}

In [140]:
sagemaker_client.list_endpoints()

{'Endpoints': [{'EndpointName': 'cls-e41ed1f6-f8f9-4adc-9afd-335ab04e08bd',
   'EndpointArn': 'arn:aws:sagemaker:us-east-1:298748835671:endpoint/cls-e41ed1f6-f8f9-4adc-9afd-335ab04e08bd',
   'CreationTime': datetime.datetime(2026, 2, 23, 2, 21, 40, 313000, tzinfo=tzlocal()),
   'LastModifiedTime': datetime.datetime(2026, 2, 23, 2, 25, 59, 181000, tzinfo=tzlocal()),
   'EndpointStatus': 'InService'},
  {'EndpointName': 'keypoint-regression-e41ed1f6-f8f9-4adc-9afd-335ab04e08bd',
   'EndpointArn': 'arn:aws:sagemaker:us-east-1:298748835671:endpoint/keypoint-regression-e41ed1f6-f8f9-4adc-9afd-335ab04e08bd',
   'CreationTime': datetime.datetime(2026, 2, 23, 2, 17, 4, 300000, tzinfo=tzlocal()),
   'LastModifiedTime': datetime.datetime(2026, 2, 23, 2, 21, 27, 742000, tzinfo=tzlocal()),
   'EndpointStatus': 'InService'}],
 'ResponseMetadata': {'RequestId': 'fc5a6801-d98e-41c7-852f-b9841a2d71cf',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'fc5a6801-d98e-41c7-852f-b9841a2d71cf

# Deleting endpoint

In [135]:
""" cw_client.delete_alarms(AlarmNames=["KeypointModel-MSE-Violation"])
cw_client.delete_alarms(AlarmNames=["ClsModel-Accuracy-Violation"])

kp_monitor.stop_monitoring_schedule()
kp_monitor.delete_monitoring_schedule()

cls_monitor.stop_monitoring_schedule()
cls_monitor.delete_monitoring_schedule() """

' cw_client.delete_alarms(AlarmNames=["KeypointModel-MSE-Violation"])\ncw_client.delete_alarms(AlarmNames=["ClsModel-Accuracy-Violation"])\n\nkp_monitor.stop_monitoring_schedule()\nkp_monitor.delete_monitoring_schedule()\n\ncls_monitor.stop_monitoring_schedule()\ncls_monitor.delete_monitoring_schedule() '

In [136]:
""" sagemaker_client.delete_endpoint(EndpointName=kp_endpoint_name)
sagemaker_client.delete_endpoint(EndpointName=cls_endpoint_name) """

' sagemaker_client.delete_endpoint(EndpointName=kp_endpoint_name)\nsagemaker_client.delete_endpoint(EndpointName=cls_endpoint_name) '